<a href="https://colab.research.google.com/github/hwsinha/Brain-Tumour-Detector/blob/main/unet_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Setup Kaggle & Download BraTS Dataset

In [4]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"hwsinha","key":"9eaf6bdf0194da6a4d8e4d87accf2aa1"}'}

In [5]:
!pip install -U kaggle -q
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d dschettler8845/brats-2021-task1
!unzip -q brats-2021-task1.zip -d /content/drive/MyDrive/brats-data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 5.9 MB/s eta 0:00:00
Dataset URL: https://www.kaggle.com/datasets/dschettler8845/brats-2021-task1
License(s): copyright-authors
100% 12.3G/12.3G [01:58<00:00, 111MB/s]



# Extract Dataset

In [6]:
!pip install nibabel -q
import os

data_root = '/content/drive/MyDrive/brats-data'
print(os.listdir(data_root)[:5])

['BraTS2021_00495.tar', 'BraTS2021_00621.tar', 'BraTS2021_Training_Data.tar']


In [7]:
import tarfile
import os

data_root = '/content/drive/MyDrive/brats-data'
extract_path = '/content/drive/MyDrive/brats-data/extracted'
os.makedirs(extract_path, exist_ok=True)

# Extract the main training data archive
with tarfile.open(os.path.join(data_root, 'BraTS2021_Training_Data.tar')) as tar:
    tar.extractall(path=extract_path)

print(os.listdir(extract_path)[:5])

/tmp/ipykernel_7581/1618536502.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_path)


['.DS_Store', 'BraTS2021_00000', 'BraTS2021_00002', 'BraTS2021_00003', 'BraTS2021_00005']


# Imports Libraries and Tools

In [8]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

# Data Visualization

In [11]:
all_entries = os.listdir(extract_path)
patient_folders = [f for f in all_entries if os.path.isdir(os.path.join(extract_path, f)) and not f.startswith('.')]
print(f"Found {len(patient_folders)} patient folders")
print(patient_folders[:5])

sample_patient = patient_folders[0]
patient_path = os.path.join(extract_path, sample_patient)
print(os.listdir(patient_path))
#tally patient folders and info

Found 1251 patient folders
['BraTS2021_00000', 'BraTS2021_00002', 'BraTS2021_00003', 'BraTS2021_00005', 'BraTS2021_00006']
['BraTS2021_00000_flair.nii.gz', 'BraTS2021_00000_seg.nii.gz', 'BraTS2021_00000_t1.nii.gz', 'BraTS2021_00000_t1ce.nii.gz', 'BraTS2021_00000_t2.nii.gz']


In [ ]:
#load flair and seg. mask then proceed